# 使用 LoRA 微调大语言模型（Hugging Face TRL）

## 什么是 LoRA？

LoRA（Low-Rank Adaptation，低秩适配）是一种**参数高效微调（PEFT）**技术。传统全量微调需要更新模型所有参数（成本极高），而 LoRA 的核心思路是：

- **冻结预训练权重**：基础模型参数完全不动，只在旁边"挂"一个轻量 adapter
- **低秩矩阵近似**：对于权重矩阵 W（m×n），学习两个小矩阵 B（m×r）和 A（r×n），用 **ΔW = B×A** 来近似参数更新
- **极少可训练参数**：因为 r ≪ min(m,n)，参数量通常减少 90% 以上
- **效果接近全量微调**：在绝大多数任务上性能损失极小

**关键初始化细节**：训练开始时 B 全部初始化为 0，A 随机初始化（高斯分布）。因此初始 ΔW = B×A = 0，模型行为与原始预训练模型完全一致，不会破坏预训练权重——这是 LoRA 能稳定训练的核心保证。

**推理时**：W_effective = W_frozen + (alpha/r) × B×A，无需改变模型结构。

## 本笔记学习目标

1. 理解 LoRA 各关键参数（rank、alpha、dropout、target_modules）的含义
2. 使用 `trl` 的 `SFTTrainer` + `peft` 的 `LoraConfig` 完成监督微调（SFT）
3. **对比微调前后效果**：训练前保存基础模型回答，训练后用同一问题集做直观对比

## 流程概览

```
加载基础模型 → 记录基础模型回答（基准）→ 配置 LoRA → 配置训练参数
→ 构建 SFTTrainer → 启动训练 → 合并 Adapter（可选）→ 加载微调后模型 → 效果对比
```

## 1. 环境配置

安装必要的 HuggingFace 生态库：

| 库 | 用途 |
|----|------|
| `transformers` | 模型加载、tokenizer、推理 pipeline |
| `datasets` | 数据集加载与处理 |
| `trl` | SFT 训练框架（SFTTrainer） |
| `peft` | LoRA/Adapter 配置与管理 |
| `huggingface_hub` | 模型上传、下载、登录认证 |

In [ ]:
# 安装依赖（Google Colab 中取消注释）
# !pip install transformers datasets trl huggingface_hub peft

# 登录 HuggingFace Hub（用于下载模型/上传结果）
from huggingface_hub import login

login()
# 也可以设置环境变量 HF_TOKEN 来避免每次手动登录

## 2. 加载数据集

本笔记使用 `HuggingFaceTB/smoltalk` 数据集的 `everyday-conversations` 子集，包含 2260 条日常多轮对话。

每条样本的核心字段是 `messages`，结构如下：

```json
{
  "messages": [
    {"role": "user",      "content": "Hi! How are you?"},
    {"role": "assistant", "content": "I'm doing well, thanks for asking!"}
  ]
}
```

`SFTTrainer` 会自动读取 `messages` 字段，通过 `SFTConfig.chat_template_path` 指定的模板将其格式化为训练文本（如 `<｜User｜>Hi!<｜Assistant｜>I'm doing well...`），无需手动预处理。


In [5]:
from datasets import load_dataset

# 加载 smoltalk 数据集中的 everyday-conversations 子集
# 该数据集包含日常对话，格式为多轮 messages（role + content）
# 用于监督微调（SFT），教模型以自然对话方式回应用户
dataset = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations")
dataset, dataset["train"][0]

(DatasetDict({
     train: Dataset({
         features: ['full_topic', 'messages'],
         num_rows: 2260
     })
     test: Dataset({
         features: ['full_topic', 'messages'],
         num_rows: 119
     })
 }),
 {'full_topic': 'Travel/Vacation destinations/Beach resorts',
  'messages': [{'content': 'Hi there', 'role': 'user'},
   {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
   {'content': "I'm looking for a beach resort for my next vacation. Can you recommend some popular ones?",
    'role': 'user'},
   {'content': "Some popular beach resorts include Maui in Hawaii, the Maldives, and the Bahamas. They're known for their beautiful beaches and crystal-clear waters.",
    'role': 'assistant'},
   {'content': 'That sounds great. Are there any resorts in the Caribbean that are good for families?',
    'role': 'user'},
   {'content': 'Yes, the Turks and Caicos Islands and Barbados are excellent choices for family-friendly resorts in the Caribbean. They offer

## 3. LoRA 微调训练

LoRA 训练只需三步配置：`LoraConfig`（adapter 结构）→ `SFTConfig`（训练超参数）→ `SFTTrainer`（组装并启动）。

各方案显存需求对比：

| 方案 | 可训练参数比例 | 显存需求 | 典型场景 |
|------|-------------|---------|---------|
| 全量微调（Full FT） | 100% | 极高（需多卡） | 充裕计算资源 |
| LoRA | ~1–5% | 低 | 单卡微调大模型 |
| QLoRA（4bit + LoRA） | ~1–5% | 极低 | 消费级 GPU |

### 3.1 加载基础模型

加载 `DeepSeek-R1-Distill-Qwen-1.5B` 因果语言模型和对应的 tokenizer。该模型基于 Qwen2.5 架构，已内置 chat template，`SFTConfig` 通过 `chat_template_path` 参数指定，由 `SFTTrainer` 自动处理，无需手动调用 `setup_chat_format`。

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer
import torch

# 自动选择最优设备：优先 GPU(CUDA)，其次 Apple Silicon(MPS)，最后 CPU
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"使用设备: {device}")

# 加载基础模型（DeepSeek-R1-Distill-Qwen-1.5B 是 DeepSeek-R1 蒸馏的 1.5B 参数因果语言模型）
# 基于 Qwen2.5 架构，已内置 chat template，支持 apply_chat_template
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# 精度选择：
# - CUDA：使用 bfloat16（与 DeepSeek-R1 训练精度一致，相比 fp32 显存减半）
# - MPS / CPU：MPS 不支持 bfloat16，回退到 float32
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name,
    torch_dtype=torch_dtype,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# 微调后模型的本地保存目录名
finetune_name = "DeepSeek-R1-FT-MyDataset"
# finetune_tags 仅在 push_to_hub=True 时作为 Hub 仓库 metadata 使用
# 当前 push_to_hub=False，此变量不影响训练，仅作记录
finetune_tags = ["smol-course", "module_1"]


使用设备: mps


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### 3.2 记录基础模型回答（微调前基准）

在 LoRA 微调开始之前，先用测试问题集跑一遍基础模型并保存回答（`base_results`）。
训练结束后用同一批问题测试微调后的模型，两组回答直接对比，即可看出微调效果。

> **关于 DeepSeek-R1-Distill 的 thinking token**：
> 该模型通过链式推理（CoT）蒸馏训练，回复前可能先输出 `<think>…</think>` 思考链，再给出最终答案。
> 微调前后这一行为可能发生变化（日常对话数据集通常不含思考链），**这本身也是一个有趣的观察维度**。


In [7]:
from transformers import pipeline

# 定义用于对比测试的提示词（覆盖不同类型：知识问答、代码生成、数学推理、概念辨析）
TEST_PROMPTS = [
    "What is the capital of Germany? Explain why that's the case and if it was different in the past?",
    "Write a Python function to calculate the factorial of a number.",
    "A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?",
    "What is the difference between a fruit and a vegetable? Give examples of each.",
]

def run_inference(pipe, prompt, max_new_tokens=200):
    """
    使用 pipeline 对单条 prompt 做推理，返回模型生成的文本。
    apply_chat_template 将 prompt 包装成模型期望的对话格式（自动适配各模型的 chat template），
    add_generation_prompt=True 在末尾添加助手回复的起始标记，引导模型开始生成。
    """
    formatted = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = pipe(formatted, max_new_tokens=max_new_tokens)
    # 截掉输入部分，只保留模型新生成的文本
    return outputs[0]["generated_text"][len(formatted):].strip()

# 用 base 模型（微调前）跑一遍测试，把结果保存到 base_results
# 注意：base_results 是一个普通 Python dict，在 kernel 运行期间会一直保留在内存中
# 训练结束后仍可用它与微调后的模型做对比
print("=" * 60)
print("【基础模型（微调前）的回答】")
print("=" * 60)

base_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

base_results = {}
for prompt in TEST_PROMPTS:
    response = run_inference(base_pipe, prompt)
    base_results[prompt] = response
    print(f"\n问题: {prompt}")
    print(f"基础模型回答: {response}")
    print("-" * 60)

# ── 推理完成后立即释放 base_pipe，避免训练阶段同时占用两份 GPU 显存 ──
del base_pipe
torch.cuda.empty_cache()
print("\n基础模型推理完成，显存已释放，准备开始 LoRA 训练...")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


【基础模型（微调前）的回答】


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



问题: What is the capital of Germany? Explain why that's the case and if it was different in the past?
基础模型回答: Okay, so I need to figure out the capital of Germany and explain why it's there. I remember hearing that Germany is often called the "Kingdom of the East," which probably means the capital is significant for that region. But I'm not entirely sure where exactly it is. I think it's in the southeast, maybe in the Rhineland region? I've heard of cities like Berlin, Munich, and Munich again. Wait, Munich is both the capital and the largest city, right? So that must be the capital.

Now, why is the capital important? Well, it's the administrative center, so it handles all the government and business activities. It's also a major economic hub, so industries there are probably big. Plus, the capital is often associated with culture and history, so people there might have a rich history and traditions.

As for past changes, I know that in the past, Germany used to be part of the Ottoman

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



问题: Write a Python function to calculate the factorial of a number.
基础模型回答: Okay, I need to write a Python function to calculate the factorial of a number. Let's think about how to approach this.

First, I remember that the factorial of a number n, denoted as n!, is the product of all positive integers from 1 to n. So, for example, 5! is 5 × 4 × 3 × 2 × 1, which equals 120.

I should start by considering the base cases. If the input number is 0 or 1, the factorial is 1 because 0! and 1! are both 1. So, I'll handle these cases first.

Next, for numbers greater than 1, I'll need to compute the product iteratively. I can initialize a result variable to 1. Then, I'll loop from 1 up to the number, multiplying each step. For each i in this range, I'll multiply the current result by i.

I should also think
------------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



问题: A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?
基础模型回答: To determine the amount of fencing needed for the rectangular garden, I'll use the formula for the perimeter of a rectangle. The perimeter is calculated by adding together twice the length and twice the width.

First, I'll multiply the length of 25 feet by 2, which gives me 50 feet.

Next, I'll multiply the width of 15 feet by 2, resulting in 30 feet.

Finally, I'll add the two results together: 50 feet plus 30 feet equals 80 feet.

Therefore, you will need 80 feet of fencing to enclose the entire garden.
</think>

To determine how many feet of fencing are needed to enclose a rectangular garden, we can use the formula for the **perimeter** of a rectangle:

\[
\text{Perimeter} = 2 \times (\text{Length} + \text{Width})
\]

**Given:**
- **Length** of the garden =
----------------------------------------------------

### 3.3 配置 LoRA 参数（LoraConfig）

定义低秩适配矩阵的关键超参数，决定 adapter 的表达能力与参数效率。代码注释中包含每个参数的详细含义。

In [8]:
from peft import LoraConfig

# ──────────────────────────────────────────────────────────────
# LoRA 原理简述：
# 对于权重矩阵 W（形状 m×n），LoRA 不直接更新 W，而是学习两个小矩阵：
#   B（m×r）初始化为 0；A（r×n）随机初始化（高斯）
# 训练开始时 ΔW = B×A = 0，与原始模型等价，训练过程中逐步学习任务相关偏移量
# 推理时：W_effective = W_frozen + (alpha/r) × B×A
# 参数量：m×n → r×(m+n)，节省约 90%+（以 r=8, m=n=4096 为例：节省 99.6%）
# ──────────────────────────────────────────────────────────────

# r（rank，秩）：LoRA 矩阵的秩维度
# - 决定了低秩矩阵 B、A 的中间维度大小
# - 越大 → 表达能力越强，但参数量和显存占用也越多
# - 越小 → 压缩比越高，适合简单任务或资源受限场景
# - 典型范围：4（极致压缩）~ 64（高表达力）；通常用 8 或 16
rank_dimension = 8

# lora_alpha（缩放因子）：控制 LoRA 更新的强度
# - 实际缩放公式：scale = lora_alpha / r
# - 相当于调节"LoRA 对原始权重影响力"的旋钮
# - 常见设置：等于 r（scale=1，更新幅度适中）或 2×r（scale=2，更强的适应力）
# - 这里 alpha=8, r=8 → scale=1，LoRA 更新与基础权重等比例贡献
lora_alpha = 8

# lora_dropout：LoRA 层的 dropout 概率
# - 在训练时随机将一部分 LoRA 激活置零，起到正则化作用
# - 防止 adapter 过拟合到训练集
# - 数据量充足（>10k 样本）时可设为 0.0；数据较少时建议 0.05~0.1
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    # bias：是否也训练 bias 参数
    # "none" → 只训练 LoRA 矩阵，bias 完全冻结（最节省参数，最常用）
    # "all" → 所有 bias 都可训练
    # "lora_only" → 只训练被 LoRA 应用到的层的 bias
    bias="none",
    # target_modules：指定对哪些子模块应用 LoRA
    # "all-linear" → 对所有线性层（Q、K、V、O、FFN 的 up/down/gate 等）都加 LoRA
    # 也可以明确指定层名，如 ["q_proj", "v_proj"]（参数更少，效果略差）
    # 不同模型的层名不同，可用 model.named_modules() 查看
    target_modules="all-linear",
    # task_type：告知 PEFT 这是因果语言模型（自回归生成任务）
    # 影响 PEFT 内部如何处理模型输出和 loss
    task_type="CAUSAL_LM",
)

### 3.4 配置训练超参数（SFTConfig）

配置学习率、批次大小、序列长度、packing、优化器、精度等训练参数。代码注释中包含每个参数的详细说明和选取依据。

> **注意**：在 TRL 新版本中，`packing`、`max_seq_length`、`dataset_kwargs` 已从 `SFTTrainer` 迁移到 `SFTConfig`，直接传给 `SFTTrainer` 会报 `TypeError`。

In [ ]:
# SFTConfig 继承自 HuggingFace TrainingArguments，专为 SFT 场景设计
args = SFTConfig(
    # ── 输出 ──────────────────────────────────────────────────
    # 模型 checkpoint 保存到本地哪个目录
    output_dir=finetune_name,

    # ── Chat Template ─────────────────────────────────────────
    # 从指定模型克隆 chat template 并自动配置 tokenizer（替代旧版 setup_chat_format）
    # TRL v0.26+ 推荐用法：直接在 SFTConfig 中声明，SFTTrainer 会自动处理 special token
    chat_template_path=model_name,

    # ── 训练轮数 ───────────────────────────────────────────────
    # 遍历完整训练集的次数；1 轮通常用于快速验证，正式训练可设 3~5
    num_train_epochs=1,

    # ── 批次大小 ───────────────────────────────────────────────
    # 每张 GPU 每步处理的样本数；受显存限制，通常设为 1~4
    per_device_train_batch_size=1,
    # 梯度累积步数：每 N 步才做一次参数更新
    # 等效 batch size = per_device_train_batch_size × gradient_accumulation_steps × GPU 数量
    # 这里等效 batch = 1 × 4 = 4，1.5B 模型显存占用更大，减小单步 batch
    gradient_accumulation_steps=4,

    # ── 序列长度与 Packing ──────────────────────────────────────
    # max_seq_length：截断/拼接序列的最大长度（token 数）
    # 超过此长度的样本会被截断；packing 时多个短样本会被拼接以填满此长度
    max_seq_length=1512,
    # packing：将多条短样本拼接填满 max_seq_length，避免大量 padding
    # 对话数据集中样本通常较短，packing 可将 GPU 利用率从约 30% 提升到 90%+
    packing=True,
    # dataset_kwargs：数据集预处理参数（packing 时生效）
    dataset_kwargs={
        # add_special_tokens=False：chat_template 已处理特殊 token，无需重复添加
        "add_special_tokens": False,
        # append_concat_token=False：packing 时不在样本间插入额外分隔符
        "append_concat_token": False,
    },

    # ── 显存优化 ───────────────────────────────────────────────
    # gradient_checkpointing：以重计算换显存
    # 正向传播时不保存中间激活值，反向传播时重新计算；节省约 30-40% 显存，但速度变慢
    gradient_checkpointing=True,

    # ── 优化器 ────────────────────────────────────────────────
    # adamw_torch_fused：PyTorch >= 2.0 的融合版 AdamW，将多个 CUDA kernel 合并执行
    # 相比普通 AdamW 快约 10-15%，且精度相同（如报错，可改为 "adamw_torch"）
    # adamw_torch_fused 仅支持 CUDA；MPS/CPU 环境自动回退到 adamw_torch
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    # 学习率：LoRA 微调推荐值比全量微调大（因为只更新少量参数，影响范围有限）
    learning_rate=2e-4,
    # 梯度裁剪阈值：防止梯度爆炸；将梯度范数限制在 0.3 以内
    max_grad_norm=0.3,

    # ── 学习率调度 ─────────────────────────────────────────────
    # warmup_ratio：热身阶段占总步数的比例
    # 前 3% 的步骤学习率从 0 线性增长到 learning_rate，避免初始大梯度破坏预训练权重
    warmup_ratio=0.03,
    # 热身结束后保持学习率不变（适合短时间微调；长时间训练可改为 "cosine"）
    lr_scheduler_type="constant",

    # ── 日志与保存 ─────────────────────────────────────────────
    logging_steps=10,       # 每 10 步打印一次 loss 等指标
    save_strategy="epoch",  # 每个 epoch 结束时保存 checkpoint

    # ── 精度 ──────────────────────────────────────────────────
    # bf16：bfloat16 混合精度训练，相比 fp32 显存减半、速度加倍
    # 需要 Ampere 架构 GPU（A10G/A100/RTX 3090+）；T4/V100 不支持，会自动回退到 fp32
    bf16=torch.cuda.is_bf16_supported(),

    # ── 其他 ──────────────────────────────────────────────────
    push_to_hub=False,   # 不自动上传到 HuggingFace Hub
    report_to="none",    # 不接入 W&B / TensorBoard 等实验追踪平台
)

### 3.5 构建训练器（SFTTrainer）

将模型、数据集、LoRA 配置和训练参数组合为 `SFTTrainer`。传入 `peft_config` 后，训练器会自动冻结基础模型并注入 LoRA 矩阵。序列长度、packing 等参数已在 `SFTConfig` 中统一配置，这里只需传入核心组件。最后打印可训练参数量，直观感受 LoRA 的参数效率。

In [ ]:
# SFTTrainer 在 Trainer 基础上增加了以下能力：
# 1. 自动将数据集 messages 字段通过 chat_template 格式化为训练文本
# 2. 原生集成 PEFT（只需传入 peft_config，剩下的自动处理）
# 3. 支持 packing、max_seq_length 等（已在 SFTConfig/args 中统一配置）
trainer = SFTTrainer(
    model=model,
    args=args,                        # 包含所有训练超参数（含 packing、max_seq_length、dataset_kwargs）
    train_dataset=dataset["train"],
    # 传入 LoRA 配置后，SFTTrainer 会自动：
    # - 冻结 base model 所有参数（requires_grad=False）
    # - 在 target_modules 上注入可训练的 LoRA 矩阵（B 初始化为 0）
    # - 训练时只更新这些 LoRA 参数
    peft_config=peft_config,
    tokenizer=tokenizer,
)

# 打印可训练参数量，直观感受 LoRA 的参数效率
# LoRA 正确注入后，可训练比例通常在 0.5%~5% 之间
# 如果显示接近 100%，说明 peft_config 未生效，需检查配置
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"可训练参数: {trainable:,} / 总参数: {total:,} ({100 * trainable / total:.2f}%)")

### 3.6 启动训练

调用 `trainer.train()` 开始训练。由于使用 PEFT，训练结束后 `save_model()` 只保存 adapter 权重（通常几 MB），而非完整模型（几 GB）。

In [ ]:
# 启动训练
trainer.train()

# 保存 adapter 权重到 output_dir（即 finetune_name 目录）
# 由于使用 PEFT，save_model() 只保存 adapter 文件（几 MB），而非完整模型（几 GB）
# 保存内容：adapter_config.json、adapter_model.safetensors、tokenizer_config.json 等
# Section 5 将从此目录重新加载微调后的模型
trainer.save_model()

> **训练耗时参考**：本笔记使用 DeepSeek-R1-Distill-Qwen-1.5B（1.5B 参数）+ 2260 条对话训练 1 epoch，在消费级 GPU（如 T4）上约需 **15~30 分钟**（相比 135M 小模型慢约 10x）。在 A10G（`g5.2xlarge`，$1.21/h）上约需 **5~10 分钟**。如需更好效果，可将 `num_train_epochs` 改为 3。

## 4. （可选）合并 LoRA Adapter

训练完成后，adapter 权重（仅几 MB）与基础模型是分离存储的。根据使用场景选择部署方式：

| 部署方式 | 优点 | 缺点 | 适用场景 |
|---------|------|------|---------|
| **分离保存**（adapter + base） | 可随时切换多个 adapter、可继续训练 | 推理时有额外计算开销 | 多任务、科研实验 |
| **合并后部署**（`merge_and_unload`） | 推理更快、部署简单、框架兼容性好 | 无法再拆分 adapter | 生产上线 |

`merge_and_unload()` 将 ΔW = B×A 的计算结果加回原始权重 W，返回一个普通的 `transformers` 模型。

> **注意**：合并后的模型保存到独立目录（`finetune_name + "-merged"`），**不会覆盖** adapter 目录（`finetune_name`），Section 5 仍可正常从 adapter 目录加载进行效果对比。

In [ ]:
from peft import AutoPeftModelForCausalLM

# 用独立变量名加载，避免覆盖训练时的 model 变量
# low_cpu_mem_usage=True：在 CPU 上逐层加载，减少峰值内存
peft_model_to_merge = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=args.output_dir,  # adapter 目录
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

# merge_and_unload()：将 ΔW = B×A 加回冻结的 W，返回普通 transformers 模型
# 合并后 adapter 矩阵消失，模型结构与原始基础模型完全相同，可直接用 AutoModelForCausalLM 加载
merged_model = peft_model_to_merge.merge_and_unload()

# 释放 PEFT 模型占用的内存，只保留合并后的模型
del peft_model_to_merge
torch.cuda.empty_cache()

# 保存到独立目录，不覆盖 adapter 目录（Section 5 仍需要从 adapter 目录加载）
merged_save_dir = finetune_name + "-merged"
merged_model.save_pretrained(merged_save_dir, safe_serialization=True, max_shard_size="2GB")
# tokenizer 与模型权重强耦合（embedding 大小必须匹配），必须一起保存
tokenizer.save_pretrained(merged_save_dir)

print(f"合并后模型已保存到本地目录: {merged_save_dir}/")
print("可用 AutoModelForCausalLM.from_pretrained(merged_save_dir) 直接加载")

## 5. 效果对比：微调前 vs 微调后

**对比方式说明**：
- `base_results`：在 Section 3.2 中用基础模型（训练前）跑的回答，存储在内存中
- `ft_results`：用从本地 adapter 目录重新加载的微调后模型跑的回答

两者使用完全相同的测试问题和推理函数（`run_inference`），直接对比输出即可看出微调效果。

> **关于 `push_to_hub=False`**：模型保存在本地目录 `finetune_name/`（即 `DeepSeek-R1-FT-MyDataset/`），`AutoPeftModelForCausalLM.from_pretrained()` 会优先查找本地路径，无需从 Hub 下载。

### 5.1 加载微调后模型

释放训练阶段的显存，加载保存的 LoRA adapter，构建用于推理的 pipeline。

In [ ]:
# 释放训练阶段占用的所有显存，避免加载 ft_model 时 OOM
del trainer
del model        # 训练时的 LoRA 模型（含 adapter 权重，已 save_model() 到磁盘）

# 如果运行了 Section 4（可选合并），merged_model 也可能还在内存中
try:
    del merged_model
except NameError:
    pass

torch.cuda.empty_cache()
print("显存已释放，准备加载微调后模型...")

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

# 从本地 adapter 目录加载微调后的模型
# finetune_name 目录结构：adapter_config.json + adapter_model.safetensors + tokenizer 文件
# AutoPeftModelForCausalLM 会先从 HF 缓存加载基础模型，再挂载 LoRA adapter
tokenizer = AutoTokenizer.from_pretrained(finetune_name)
ft_model = AutoPeftModelForCausalLM.from_pretrained(
    finetune_name,
    device_map="auto",      # 自动分配到可用 GPU/CPU
    torch_dtype=torch.bfloat16,
)

# 构建微调后模型的推理 pipeline
# 注意：model 已通过 device_map="auto" 放置到设备，pipeline 无需再指定 device
ft_pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=tokenizer,
)
print("微调后模型加载完成，开始对比推理...")

### 5.2 输出对比结果

用微调后的模型跑同一批测试问题，与 Section 3.2 保存的 `base_results` 对比。重点观察：
- 微调后是否更好地遵循了对话格式（assistant 角色风格）
- 指令跟随能力是否有提升（直接回答 vs 重复 prompt/无意义续写）
- 回复风格是否向 everyday-conversations 数据集的风格靠拢（简洁口语化）

In [ ]:
# 防御性检查：base_results 在 Section 3.2 中产生，若重启 kernel 后直接运行此 cell 会报错
assert "base_results" in globals(), (
    "base_results 未定义！请先运行 Section 3.2（cell 9）跑基础模型推理，再运行此 cell。"
)

print("=" * 60)
print("【微调前 vs 微调后 回答对比】")
print("=" * 60)

ft_results = {}
for prompt in TEST_PROMPTS:
    ft_response = run_inference(ft_pipe, prompt)
    ft_results[prompt] = ft_response

    print(f"\n{'='*60}")
    print(f"问题: {prompt}")
    print(f"\n【微调前（base model）】\n{base_results[prompt]}")
    print(f"\n【微调后（LoRA adapter）】\n{ft_response}")

print("\n" + "=" * 60)
print("【观察要点】")
print("""
微调前后的主要差异通常体现在：

1. 回复格式：
   - 微调前：模型可能不遵循对话格式，输出杂乱或截断
   - 微调后：遵循 assistant 角色，给出结构清晰的回复

2. 话题相关性（针对 everyday-conversations 数据集）：
   - 微调后的模型更擅长日常对话风格的表达
   - 回复更简洁、口语化，而非生硬的知识堆砌

3. 指令跟随能力：
   - 微调后能更准确地理解并回应用户问题
   - 避免重复 prompt 内容或产生无意义续写

如果效果差异不明显，可尝试：
  → num_train_epochs=3（更多训练轮次）
  → rank_dimension=16（更强的 adapter 表达力）
  → 换用更大规模训练数据（如加载 smoltalk 完整数据集而非 everyday-conversations 子集）
  → 换用更强基础模型（如 DeepSeek-R1-Distill-Qwen-7B）
""")

## 6. 各 GPU 配置训练时间估算

### 当前训练规模

| 维度 | 配置 |
|------|------|
| 模型 | DeepSeek-R1-Distill-Qwen-1.5B（15 亿参数） |
| LoRA | r=8，target_modules="all-linear" |
| 数据集 | smoltalk everyday-conversations，2260 条对话 |
| 序列处理 | packing=True，max_seq_length=1512 |
| 有效训练步数 | 约 75 个 optimizer steps（packing 后约 300 条序列 / batch_size=1 / gradient_accumulation=4，1 epoch） |
| 精度 | bf16（Ampere 架构及以上支持） |

---

### 各阶段耗时对比

| 阶段 | T4（参考基线） | RTX 5090 32G × 1 | RTX 5090 32G × 4 | H100 SXM 80G | H200 SXM 141G |
|------|------------|-----------------|-----------------|-------------|--------------|
| 数据集 tokenize/packing（CPU） | ~30s | ~30s | ~30s | ~30s | ~30s |
| 模型加载 | ~30s | ~15s | ~15s | ~10s | ~10s |
| 基础模型推理（4 条测试） | ~30s | ~10s | ~10s | ~8s | ~8s |
| **LoRA 训练（1 epoch）** | **15–30 分钟** | **3–5 分钟** | **1–2 分钟** | **2–4 分钟** | **2–3 分钟** |
| 保存 / 合并 / ft 推理 | ~60s | ~30s | ~30s | ~30s | ~30s |
| **全流程合计** | **20–35 分钟** | **5–8 分钟** | **3–4 分钟** | **4–6 分钟** | **3–5 分钟** |

---

### 关键说明

**5090 × 4 卡并不比单卡快太多**

1.5B 模型计算量相比 135M 大约 10x，但 DDP 的 AllReduce 梯度同步通信开销依然会占据一定比例。实际加速比约为 2.5–3x，而非理论 4x。

**H100 SXM vs RTX 5090**

H100 SXM 峰值 BF16 算力（~989 TFLOPS dense）高于 5090（~600–700 TFLOPS dense）。对 1.5B 模型，H100 的计算优势已开始显现，但内存带宽仍是主要瓶颈。

**H200 vs H100**

H200 与 H100 SXM 计算核心相同，主要升级是内存容量（80G HBM3 → 141G HBM3e）和带宽（3.35 TB/s → 4.8 TB/s）。对 1.5B 模型，H200 的带宽优势在大 batch 时会有所体现。

**总结**

1.5B 模型 + 2260 条数据 + 1 epoch 在高端 GPU（A10G/H100）上为 **分钟级别**，T4 等消费级 GPU 约需 **15–30 分钟**。若要进一步体现 GPU 性能差距，可换用更大规模数据集或更多 epoch。


## 7. 学习问答记录

### Q1：packing 多条对话会影响模型学习效果吗？

**问题背景**：多条原始对话被 packing 为一条训练序列，而这些对话可能在说完全不同的事情，模型 attention 是否会产生跨对话的错误关联？

**根源分析**

Transformer 的 attention 对整个序列计算。如果把对话 A 和对话 B 直接拼接，模型处理对话 B 的 token 时，attention 会"看到"对话 A 的内容，产生错误的跨对话上下文。

**解决方案：attention mask（document masking）**

TRL 的 `SFTTrainer` 在 packing 时会自动生成 **position ids** 和 **attention mask**，确保每条对话只能 attend 自身内部的 token，跨对话的 attention 被 mask 掉：

```
[对话A token1, token2, token3 | 对话B token1, token2 | 对话C ...]
 ↑_____attention 只在 A 内部____↑  ↑___只在 B 内部___↑
```

packing 只是"物理上拼在一起节省显存"，逻辑上每条对话仍然独立训练。

**何时仍有轻微影响**

如果不使用 flash attention（普通 attention + packed sequence），attention mask 的实现成本很高，部分框架会退化为允许跨对话 attend，此时才会有真实的学习质量下降。TRL 默认使用了正确的 document masking 实现，本 notebook 无需担心这个问题。

---

### Q2：实际 LoRA 微调时，如何决策 `target_modules` 参数的选择？

**问题背景**：`target_modules="all-linear"` 会覆盖所有线性层，但有时会看到只选 `["q_proj", "v_proj"]` 的配置，两者有何区别？实践中如何选择？

**`"all-linear"` 的含义**

覆盖模型中**所有 `nn.Linear` 层**，包括：
- Attention 层：`q_proj`, `k_proj`, `v_proj`, `o_proj`
- FFN 层：`gate_proj`, `up_proj`, `down_proj`（LLaMA/Qwen 系列）
- LM head（视实现而定）

覆盖最广、参数量最多，训练成本也最高。

**决策框架**

| 场景 | 推荐配置 | 理由 |
|------|----------|------|
| 资源有限 / 快速验证 | `["q_proj", "v_proj"]` | 原始 LoRA 论文的最小配置，大多数任务已够用 |
| 指令跟随 / 对话风格调整 | `["q_proj", "k_proj", "v_proj", "o_proj"]` | 完整覆盖 attention，调整模型"如何表达" |
| 领域知识注入 | Attention 全部 + FFN 层 | FFN 层存储"事实知识"，学新知识需覆盖 FFN |
| 追求最优 / 资源充足 | `"all-linear"` | 覆盖最广，不确定哪些层更重要时的保底选择 |

**`r` 与 `target_modules` 的联动**

覆盖层越多，可以适当降低 `r` 来控制总参数量：

```
总 LoRA 参数 ≈ 2 × r × (层数 × 平均维度)
```

例如 `"all-linear"` + `r=4` 的总参数量可能与 `["q_proj","v_proj"]` + `r=16` 相近，但前者覆盖更均匀，效果往往更好。

**查看当前模型可选层名**

不同架构的线性层命名不同（如 Qwen 用 `c_attn`，LLaMA 用 `q_proj`），可用以下代码确认：

```python
linear_layers = [name for name, module in model.named_modules()
                 if isinstance(module, torch.nn.Linear)]
print(linear_layers)
```